# Create and Test VI Files for the BL-AGN Candidates

Ragadeepika Pucha, Stéphanie Juneau, Arjun Dey, Yao-Yuan Mao, and others ...

This notebook contains code used to create the VI Files and for subsequent testing.

In [1]:
import numpy as np

from astropy.table import Table, Column

import sys
sys.path.append('/global/cfs/cdirs/desi/users/raga19/repos/EmFit/py/')
sys.path.append('/global/cfs/cdirs/desi/users/raga19/repos/DESI_Project/py/')

import plot_utils

import bpt
import utils
import plotting_functions as pf

import matplotlib.pyplot as plt
from matplotlib.backends import backend_pdf as pdf

In [2]:
## Making the matplotlib plots look nicer
settings = {
    'mathtext.fontset': 'stix',
    'font.family': 'STIXGeneral',
    'font.size':24,
    'axes.labelsize': 'large',
    'lines.markersize': 5,
    'axes.linewidth':2.0,
    'xtick.major.size':6.0,
    'xtick.minor.size':4.0,
    'xtick.major.width':2.0,
    'xtick.minor.width':1.5,
    'xtick.direction':'in', 
    'xtick.minor.visible':True,
    'xtick.top':True,
    'ytick.major.size':6.0,
    'ytick.minor.size':4.0,
    'ytick.major.width':2.0,
    'ytick.minor.width':1.5,
    'ytick.direction':'in', 
    'ytick.minor.visible':True,
    'ytick.right':True,
    'contour.linewidth':3.0,
    'savefig.bbox': 'tight',
    'savefig.dpi': 200,
}

plt.rcParams.update(**settings)

In [3]:
t = Table.read('../catalogs/iron-selection-v2.3.fits')
len(t)

6099572

In [4]:
## Stellar Mass related variables

logmass = t['LOGM'].data
lomass = (logmass <= 9.5)
himass = (logmass > 9.5)

len(t[lomass]), len(t[himass])

(987987, 5111585)

In [5]:
## BL Related Variables

z = t['Z'].data
ha_b = t['HA_B_FLUX'].data
ha_b_snr = ha_b/t['HA_B_FLUX_ERR'].data
ha_b_aon = t['HA_B_AMPLITUDE'].data/t['NII_HA_NOISE'].data
ha_b_sig = t['HA_B_SIGMA'].data
ha_b_sig_snr = ha_b_sig/t['HA_B_SIGMA_ERR'].data
per_broad = t['PROB_BROAD'].data

ha_b_lum = utils.Flux_to_Luminosity(ha_b*1e-17, z)
ha_b_fwhm = utils.sigma_to_fwhm(ha_b_sig)
bh_mass = utils.calculate_bh_masses(ha_b_lum, ha_b_fwhm)

In [6]:
## Number of two-component fits

is_sii_out = (t['SII6716_OUT_FLUX'].data != 0)
is_oiii_out = (t['OIII5007_OUT_FLUX'].data != 0)

print ('Number of sources with an extra [SII] component: ', len(t[is_sii_out]), round(len(t[is_sii_out])*100/len(t), 1), '%')
print ('Number of sources with an extra [OIII] component: ', len(t[is_oiii_out]), round(len(t[is_oiii_out])*100/len(t), 1), '%')

Number of sources with an extra [SII] component:  35666 0.6 %
Number of sources with an extra [OIII] component:  71618 1.2 %


In [7]:
## Double-Peaked Emission Line Flags
is_sii_dbl = t['SII_DBL_FLAG'].data
is_oiii_dbl = t['OIII_DBL_FLAG'].data

len(t[is_sii_dbl]), len(t[is_oiii_dbl])

(24735, 15702)

In [8]:
## Emission-line measurements -- Narrow only and sum

## Narrow Only
nii_n = t['NII6583_FLUX'].data
nii_snr_n = nii_n/t['NII6583_FLUX_ERR'].data
nii_aon = t['NII6583_AMPLITUDE'].data/t['NII_HA_NOISE'].data

oiii_n = t['OIII5007_FLUX'].data
oiii_snr_n = oiii_n/t['OIII5007_FLUX_ERR'].data
oiii_aon = t['OIII5007_AMPLITUDE'].data/t['OIII_NOISE'].data

ha_n = t['HA_N_FLUX'].data
ha_snr_n = ha_n/t['HA_N_FLUX_ERR'].data
ha_aon = t['HA_N_AMPLITUDE'].data/t['NII_HA_NOISE'].data

hb_n = t['HB_N_FLUX'].data
hb_snr_n = hb_n/t['HB_N_FLUX_ERR'].data
hb_aon = t['HB_N_AMPLITUDE'].data/t['HB_NOISE']


## Sum
nii_sum = t['NII6583_FLUX'].data + t['NII6583_OUT_FLUX'].data
nii_snr_sum = nii_sum/np.sqrt((t['NII6583_FLUX_ERR'].data**2) + (t['NII6583_OUT_FLUX_ERR'].data**2))

oiii_sum = t['OIII5007_FLUX'].data + t['OIII5007_OUT_FLUX'].data
oiii_snr_sum = oiii_sum/np.sqrt((t['OIII5007_FLUX_ERR'].data**2) + (t['OIII5007_OUT_FLUX_ERR'].data**2))

ha_sum = t['HA_N_FLUX'].data + t['HA_OUT_FLUX'].data
ha_snr_sum = ha_sum/np.sqrt((t['HA_N_FLUX_ERR'].data**2) + (t['HA_OUT_FLUX_ERR'].data**2))

hb_sum = t['HB_N_FLUX'].data + t['HB_OUT_FLUX'].data
hb_snr_sum = hb_sum/np.sqrt((t['HB_N_FLUX_ERR'].data**2) + (t['HB_OUT_FLUX_ERR'].data**2))

In [9]:
## Emission-line measurements

## Empty arrays
nii, nii_snr = np.zeros(len(t)), np.zeros(len(t))
oiii, oiii_snr = np.zeros(len(t)), np.zeros(len(t))
ha, ha_snr = np.zeros(len(t)), np.zeros(len(t))
hb, hb_snr = np.zeros(len(t)), np.zeros(len(t))

## [OIII] Condition
oiii[is_oiii_dbl], oiii[~is_oiii_dbl] = oiii_sum[is_oiii_dbl], oiii_n[~is_oiii_dbl]
oiii_snr[is_oiii_dbl], oiii_snr[~is_oiii_dbl] = oiii_snr_sum[is_oiii_dbl], oiii_snr_n[~is_oiii_dbl]

## [NII]+Ha Condition
nii[is_sii_dbl], nii[~is_sii_dbl] = nii_sum[is_sii_dbl], nii_n[~is_sii_dbl]
nii_snr[is_sii_dbl], nii_snr[~is_sii_dbl] = nii_snr_sum[is_sii_dbl], nii_snr_n[~is_sii_dbl]

ha[is_sii_dbl], ha[~is_sii_dbl] = ha_sum[is_sii_dbl], ha_n[~is_sii_dbl]
ha_snr[is_sii_dbl], ha_snr[~is_sii_dbl] = ha_snr_sum[is_sii_dbl], ha_snr_n[~is_sii_dbl]

hb[is_sii_dbl], hb[~is_sii_dbl] = hb_sum[is_sii_dbl], hb_n[~is_sii_dbl]
hb_snr[is_sii_dbl], hb_snr[~is_sii_dbl] = hb_snr_sum[is_sii_dbl], hb_snr_n[~is_sii_dbl]

nii_ha = np.log10(nii/ha)
oiii_hb = np.log10(oiii/hb)

In [10]:
is_agn, is_comp, is_sf = bpt.Classify_NII_BPT(nii_ha, oiii_hb)

In [11]:
## Cuts
bpt_cut = (nii_snr >= 3)&(oiii_snr >= 3)&(ha_snr >= 3)&(hb_snr >= 1)&(hb_aon >= 1)
bl_cut = (ha_b_fwhm >= 300)&(ha_b_fwhm < 15000)&(ha_b_snr >= 3)&(ha_b_sig_snr >= 3)&(ha_b_aon >= 2)&(per_broad >= 80)&(t['HA_B_SIGMA'].data > t['HA_N_SIGMA'].data)&(t['HA_N_AMPLITUDE'].data > t['HA_B_AMPLITUDE'].data)

is_bl = bpt_cut&bl_cut
is_nl = bpt_cut&~bl_cut

In [12]:
fwhm_lt1000 = (ha_b_fwhm <= 1000)

In [13]:
bpt_col = np.zeros(len(t), dtype = '<U10')
bpt_col[is_agn&bpt_cut] = 'agn'
bpt_col[is_comp&bpt_cut] = 'composite'
bpt_col[is_sf&bpt_cut] = 'sf'
bpt_col[~bpt_cut] = 'none'

In [14]:
bpt = Column(bpt_col, name = 'BPT')
t.add_column(bpt)

In [15]:
t_bl = t[is_bl&fwhm_lt1000&(is_agn|is_comp)]
t_bl.sort('LOGM')

In [16]:
len(t_bl), np.min(t_bl['LOGM'].data), np.max(t_bl['LOGM'].data)

(2309, np.float64(7.164732845920933), np.float64(11.650403448844767))

In [17]:
def create_vi_pages(table, outfile):        
    p1 = pdf.PdfPages(f'{outfile}.pdf')
    p2 = pdf.PdfPages(f'{outfile}_images.pdf')
    
    table.sort('HA_B_SIGMA', reverse = True)
    ra = table['TARGET_RA'].data
    dec = table['TARGET_DEC'].data
    tgts = table['TARGETID'].data
    logmass_sel = table['LOGM'].data
    z_sel = table['Z'].data
    ha_b_sel = table['HA_B_FLUX'].data
    ha_b_sig_sel = table['HA_B_SIGMA'].data
    bpt_sel = table['BPT'].astype(str).data
    
    ha_b_fwhm_sel = utils.sigma_to_fwhm(ha_b_sig_sel)
    ha_b_lum_sel = utils.Flux_to_Luminosity(ha_b_sel*1e-17, z_sel)
    bh_mass_sel = utils.calculate_bh_masses(ha_b_lum_sel, ha_b_fwhm_sel)
    
    f = open(f'{outfile}.txt', 'w')
    f.write('TARGETID\tVI_FLAG\n')
    
    for ii in range(len(table)):
        f.write(f'{tgts[ii]}\t 0\n')
        
        title = f'TARGETID: {tgts[ii]}; logmass: {round(logmass_sel[ii], 1)}; z: {round(z_sel[ii],2)}; log (MBH): {round(bh_mass_sel[ii],1)}; BPT Class: {bpt_sel[ii]} \n'+\
        f'https://www.legacysurvey.org/viewer-desi/desi-spectrum/daily/targetid{tgts[ii]}'
        
        f1 = plot_utils.plot_from_params(table, ii, ha_xlim = [6500, 6620], title = title)
        
        p1.savefig(f1)     
        
        imgs = pf.get_multiwavelength_cutouts(ra[ii], dec[ii], cutout_size = 30.)
        
        fig = plt.figure(figsize = (12, 3), constrained_layout = True)
        plt.suptitle(f'TARGETID: {tgts[ii]}')
        for jj, image in enumerate(imgs):
            ax = fig.add_subplot(1, 4, jj+1)
            ax.imshow(image)
            ax.set(xticks = [], yticks = [])

        plt.close()
        p2.savefig(fig)
        
    f.close()
    p1.close()
    p2.close()

In [18]:
fwhm_bins = np.arange(300, 1000, 100)
logmass_bins = [6, 9, 9.5, 10, 10.5, 11, 13]

In [19]:
ha_b_fwhm_bl = utils.sigma_to_fwhm(t_bl['HA_B_SIGMA'].data)
logmass_bl = t_bl['LOGM'].data

In [20]:
for ii in range(len(logmass_bins)-1):
    mass_min = logmass_bins[ii]
    mass_max = logmass_bins[ii+1]

    for fwhm in fwhm_bins:
        fwhm_min = fwhm
        fwhm_max = fwhm + 100

        sel = (logmass_bl >= mass_min)&(logmass_bl < mass_max)&(ha_b_fwhm_bl >= fwhm_min)&(ha_b_fwhm_bl < fwhm_max)
        tsel = t_bl[sel]

        if (len(tsel) == 0):
            continue
        elif (len(tsel) > 100):
            m_sub = np.arange(mass_min, mass_max, 0.25)

            for xx in m_sub:
                sel1 = (logmass_bl >= xx)&(logmass_bl < xx+0.25)&(ha_b_fwhm_bl >= fwhm_min)&(ha_b_fwhm_bl < fwhm_max)
                tsel1 = t_bl[sel1]

                if (len(tsel1) == 0):
                    continue
                else:
                    # print (xx, xx+0.25, fwhm_min, fwhm_max, tsel1[tsel1['TYPE'].astype(str).data == 'PSF']['TARGETID'].data)
                    filename = f'../02_Analysis/VI_Files/mass-{xx}-{xx+0.25}_fwhm-{fwhm_min}-{fwhm_max}.txt'  
                    tab = Table.read(filename, format = 'ascii')
                    print (xx, xx+0.25, fwhm_min, fwhm_max, tab['VI_FLAG'].data)
        else:
            # print (mass_min, mass_max, fwhm_min, fwhm_max, tsel[tsel['TYPE'].astype(str).data == 'PSF']['TARGETID'].data)
            filename = f'../02_Analysis/VI_Files/mass-{mass_min}-{mass_max}_fwhm-{fwhm_min}-{fwhm_max}.txt'

            tab = Table.read(filename, format = 'ascii')
            print (mass_min, mass_max, fwhm_min, fwhm_max, tab['VI_FLAG'].data)

6 9 300 400 [1 2 1 2 1 2 2 1 2 2 1 2 2 1 2 0 1 2 2 2 2 2 2 2 2 2 2]
6 9 400 500 [3 2 2 2 2 1 2 2 2 0 3 0 1 2 0 1 2 1 0]
6 9 500 600 [2 0 0 3 2 2 2 3 2 2 2]
6 9 600 700 [0 0 2 0 1 1 1 1 2]
6 9 700 800 [3 0 1 1 1 3 1 2]
6 9 800 900 [3 1 0 2 1]
6 9 900 1000 [2 1 1 2]
9 9.5 300 400 [2 2 4 4 4 4 4 4 4 2 4 4 4 4 4 4 4 4 2 4 4 2 4 4 2 1 4 4 4 0 4 4 4 1 4 4 4
 4 4 4 2 2 4 2 4 4 4 4 4 2 3 4 2 4 4 4 4 4 4 4 4 4 2 4 4 2 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 2 4 4]
9 9.5 400 500 [2 2 1 0 2 2 0 2 2 1 0 1 1 2 1 2 2 2 2 2 2 2 2 1 0 2 1 2 2 2 2 2 2 2 1 2 2
 2 2 2 3 1 1 0 2 2 1 2 1 2 2 2 2 1 1 2 2 2 1 2 2 2 1 2 2 2 1 2 1 1 2 2]
9 9.5 500 600 [0 2 0 0 0 0 0 0 0 2 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
 1 0 0 0 0 0 0 2 0 2]
9 9.5 600 700 [1 0 0 2 0 0 0 0 1 0 0 0 0 0 0 0 0 0 2 0 1 0 0 0 0 0 0 0 0 0]
9 9.5 700 800 [0 0 0 0 3 0 0 0 2 0 0 0 0 0 0 0 0 1 0 0 0]
9 9.5 800 900 [1 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 2 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1
 0 0 0]
9 9.5 900 1000 [0 0 0 0 1 1 0 0 0 0 0 0 

In [21]:
# 39627893050967666 -- good fit!

In [22]:
# for ii in range(len(logmass_bins)-1):
#     mass_min = logmass_bins[ii]
#     mass_max = logmass_bins[ii+1]

#     for fwhm in fwhm_bins:
#         fwhm_min = fwhm
#         fwhm_max = fwhm + 100

#         sel = (logmass_bl >= mass_min)&(logmass_bl < mass_max)&(ha_b_fwhm_bl >= fwhm_min)&(ha_b_fwhm_bl < fwhm_max)
#         tsel = t_bl[sel]

#         if (len(tsel) == 0):
#             continue
#         elif (len(tsel) > 100):
#             m_sub = np.arange(mass_min, mass_max, 0.25)

#             for xx in m_sub:
#                 sel1 = (logmass_bl >= xx)&(logmass_bl < xx+0.25)&(ha_b_fwhm_bl >= fwhm_min)&(ha_b_fwhm_bl < fwhm_max)
#                 tsel1 = t_bl[sel1]

#                 if (len(tsel1) == 0):
#                     continue
#                 else:
#                     outfile = f'VI_Files/mass-{xx}-{xx+0.25}_fwhm-{fwhm_min}-{fwhm_max}'
#                     print (f'------------- logmass = {xx}-{xx+0.25}; FWHM = {fwhm_min}-{fwhm_max} km/s')
#                     create_vi_pages(tsel1, outfile)
                
#         else:
#             outfile = f'VI_Files/mass-{mass_min}-{mass_max}_fwhm-{fwhm_min}-{fwhm_max}'
#             print (f'------------- logmass = {mass_min}-{mass_max}; FWHM = {fwhm_min}-{fwhm_max} km/s')
#             create_vi_pages(tsel, outfile)

---